# Nuclear Geometry and Characterization of the Quark-Gluon Plasma

<div>
<img src="https://raw.githubusercontent.com/illinois-ipaml/MachineLearningForPhysics/main/img/Project_NuclearGeometryQCD-Figure.jpg" width=600></img>
</div>

## <span style="color:Orange">Learning Objectives</span>

By the end of this project you should be able to:

* explain what the quark-gluon plasma (QGP) is and why it is studied in collisions of heavy nuclei;
* describe the nuclear density with a Woods–Saxon distribution and the geometry of a collision with an impact parameter;
* compute the eccentricity of a shape, first for a smooth overlap region and then event by event for Monte Carlo Glauber collisions;
* summarize event-by-event distributions with their moments; and
* use kernel density estimation to model the finite size of nucleons, and assess the sensitivity to the bandwidth.

## <span style="color:Orange">Overview</span>

Collisions between two nuclei at ultra-relativistic energies, at the Relativistic Heavy Ion Collider (RHIC) and the Large Hadron Collider (LHC), create the **quark-gluon plasma** (QGP), a novel state of matter in which quarks and gluons are not confined inside hadrons [[4]](https://doi.org/10.1126/science.1215901), [[5]](https://arxiv.org/abs/1802.04801). The QGP behaves like an almost perfect fluid. Its expansion converts the **shape** of the initial collision region into an anisotropic distribution of the produced particles. Measuring that anisotropy therefore tells us about the properties of the QGP, but only if we understand the initial geometry.

In this project you will learn a bit about what the QGP is and how to describe the geometry of collisions between two lead nuclei. You will start with a smooth (average) picture of the nucleus. Then you will analyse Monte Carlo Glauber simulations [[1]](https://doi.org/10.1103/PhysRevC.97.054910), [[2]](https://arxiv.org/abs/nucl-ex/0701025), in which the individual nucleons, and therefore the event-by-event fluctuations of the shape, are taken into account [[3]](https://doi.org/10.1103/PhysRevC.81.054905).

## <span style="color:Orange">Data Sources</span>

Monte Carlo Glauber simulations of Pb+Pb collisions at two fixed impact parameters $b$. Each file contains 10,000 collisions. For each collision it lists the transverse positions $(x, y)$, in fm, of every nucleon (proton or neutron) that collided with at least one nucleon from the other nucleus. These are the "participants". The columns are `event, x, y`.

* $b = 0$ fm: https://courses.physics.illinois.edu/phys498mlp/sp2025/data/projects/NuclearGeometryQGC/PbPb_b0_10kevents.csv (98 MB)
* $b = 6$ fm: https://courses.physics.illinois.edu/phys498mlp/sp2025/data/projects/NuclearGeometryQGC/PbPb_b6_10kevents.csv (59 MB)

⚠️ **Data-quality issue:** the program that produced these files wrote its progress messages into them. At the start of every 100th event, the event number is prefixed with text such as `1% done`, giving lines like `1% done100, -3.1954, 1.69976` (99 lines per file). A plain `pd.read_csv` therefore reads the `event` column as text, and grouping by it creates 99 spurious one-nucleon "events". The header also has stray spaces. The `load_glauber` function below fixes both problems.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = "https://courses.physics.illinois.edu/phys498mlp/sp2025/data/projects/NuclearGeometryQGC/"

def load_glauber(filename):
    """Read a Glauber file, removing the stray progress messages from the event column."""
    df = pd.read_csv(BASE_URL + filename, skipinitialspace=True)
    df.columns = df.columns.str.strip()
    df["event"] = df["event"].astype(str).str.replace(r"^\d+% done", "", regex=True).astype(int)
    return df

glauber = {b: load_glauber(f"PbPb_b{b}_10kevents.csv") for b in (0, 6)}
for b, df in glauber.items():
    print(f"b = {b} fm: {df['event'].nunique()} events, {len(df)} participating nucleons")

## <span style="color:Orange">Questions</span>

Answer each question in a markdown cell directly below it (add code cells where needed). For the background questions, aim for one to two well-organized paragraphs, and **cite any sources you use**. Your notebook should run from top to bottom without errors.

#### Part A: The QGP and the smooth nucleus

### <span style="color:LightGreen">Question 01</span>

What is the quark-gluon plasma? Why does it need to be created in collisions of two nuclei, rather than, for example, two protons? Ref. [[5]](https://arxiv.org/abs/1802.04801) (freely available) and Ref. [[4]](https://doi.org/10.1126/science.1215901) (available through the university library) are good starting points.

### <span style="color:LightGreen">Question 02</span>

The nuclei involved in these collisions are fully stripped of electrons. A nucleus is composed of protons and neutrons, collectively called nucleons. On average, the nucleons are distributed according to a Woods–Saxon distribution in the distance $r$ from the centre of the nucleus:

$$
\Large \rho(r) = \frac{\rho_0}{1 + \exp\left((r-R)/a\right)}
$$

For lead ($^{208}$Pb), the radius is $R = 6.62$ fm and the surface diffuseness is $a = 0.546$ fm (Table IV of Ref. [[1]](https://doi.org/10.1103/PhysRevC.97.054910); 1 fm = $10^{-15}$ m).

* Determine $\rho_0$ numerically by requiring that the nucleus contains 208 nucleons, $\int \rho(r)\, d^3r = 208$. Compare your value with the typical nuclear-matter density of about $0.16$ fm$^{-3}$.
* Plot $\rho(r)$, and overlay a uniform hard sphere of radius $R$ containing the same number of nucleons.

### <span style="color:LightGreen">Question 03</span>

Two lead nuclei collide, one moving along the $+z$ direction and the other along $-z$. When they collide, their centres are separated by a distance $b$ (the impact parameter) in the $xy$-plane. Treat the nuclei as hard spheres of radius $R = 6.62$ fm, so that their projections onto the $xy$-plane are disks. Plot the overlap region of the two disks in the $xy$-plane for $b = 0$ fm and $b = 6$ fm, with the centre of the overlap region at $x = y = 0$. What is the area of the overlap region in each case?

### <span style="color:LightGreen">Question 04</span>

The eccentricity of a distribution in the $xy$-plane can be calculated as

$$
\Large \varepsilon_2 = \frac{\sqrt{\langle r^2\cos(2\phi)\rangle^2 + \langle r^2\sin(2\phi)\rangle^2}}{\langle r^2 \rangle}
$$

where $r$ and $\phi$ are the usual polar coordinates and $\langle\cdots\rangle$ is an average over the distribution. Calculate the eccentricity of the overlap region from Question 03 for both values of $b$, assuming a uniform density inside the overlap region. (*Hint:* sample many points uniformly in the overlap region, or integrate on a fine grid.)

*Optional:* repeat the calculation weighting each point by the product of the two nuclear thicknesses $T_A(x,y)\,T_B(x,y)$, where $T(x,y) = \int \rho\,dz$ uses the Woods–Saxon density from Question 02. How does the result change, and why?

### <span style="color:LightGreen">Question 05</span>

The eccentricity defines the initial shape of the QGP, and it drives an asymmetric expansion of the QGP. Watch the public lecture ["A Look Inside the Hottest Matter in the Universe"](https://www.youtube.com/watch?v=NTfaIyInhOY) by Anne Sickles (Saturday Physics for Everyone, Illinois Physics, 2018) up to the 29-minute mark. Briefly summarize what we can learn about the QGP from the geometry of nuclear collisions.

<hr style="border:1px solid gray; margin: 2em 0;">

#### Part B: Monte Carlo Glauber collisions

The nucleus is a quantum system, and the nucleons are not static inside it. Because the nucleus is made up of individual nucleons, it is pairs of nucleons, one from each nucleus, that collide, within the smooth distribution of Question 02. The Monte Carlo Glauber model [[1]](https://doi.org/10.1103/PhysRevC.97.054910), [[2]](https://arxiv.org/abs/nucl-ex/0701025) simulates this. For each event it samples the nucleon positions in each nucleus from the Woods–Saxon distribution, places the nuclei at impact parameter $b$, and declares two nucleons to have collided if their transverse distance is small enough. The remaining questions use the simulated participant positions described in the Data Sources section.

### <span style="color:LightGreen">Question 06</span>

Plot the participant positions for a few example events at each impact parameter, overlaid on the smooth overlap region from Question 03.

Calculate $\varepsilon_2$ for each collision. First shift *each collision* so that its participants are centred at the origin ($\langle x \rangle = \langle y \rangle = 0$). You will get a different value of $\varepsilon_2$ for each collision, so 10,000 values for each file.

* Plot the distributions of $\varepsilon_2$ for $b = 0$ and $b = 6$ fm, and mark your results from Question 04.
* Calculate the mean, standard deviation and skewness of each distribution.
* Explain why $\varepsilon_2$ is non-zero at $b = 0$, and why its distribution at $b = 6$ fm differs from your smooth-geometry result in Question 04.
* Also plot the distribution of the number of participants in each event, and comment on it.

### <span style="color:LightGreen">Question 07</span>

Treating each nucleon as a point is unrealistic. Use **kernel density estimation** (KDE) to build a smooth participant density for each event that takes into account the finite size of the nucleons. The [proton radius puzzle](https://en.wikipedia.org/wiki/Proton_radius_puzzle) article discusses the measured proton charge radius (about 0.84 fm). Assume that the neutron has the same size, which is a good approximation here.

* Explain how you chose the kernel and its bandwidth. How is the bandwidth of a Gaussian kernel in two dimensions related to the root-mean-square radius of a nucleon in three dimensions?
* Calculate $\varepsilon_2$ from the KDE density (for example, by evaluating it on a grid) for each event. You may use a subset of the events if this is slow.
* How sensitive is the result to the bandwidth? Try several values.
* How does the eccentricity compare with what you found in Question 06? Write a paragraph and make plots to support your conclusions.

*Check:* for a Gaussian kernel of width $\sigma$ in each direction, you can show analytically that the numerator of $\varepsilon_2$ is unchanged, while $\langle r^2\rangle \to \langle r^2\rangle + 2\sigma^2$. Does your numerical result agree?

### <span style="color:LightGreen">Question 08</span>

Because of fluctuations, higher-order eccentricities can also be non-zero. This was first discussed in Ref. [[3]](https://doi.org/10.1103/PhysRevC.81.054905). Calculate

$$
\Large \varepsilon_3 = \frac{\sqrt{\langle r^2\cos(3\phi)\rangle^2 + \langle r^2\sin(3\phi)\rangle^2}}{\langle r^2 \rangle}
$$

for the same collisions. In general, $\varepsilon_n$ measures the magnitude of the $n\phi$ symmetry in the shape.

* What is $\varepsilon_3$ for the smooth overlap region of Question 03, and why?
* How do the values of $\varepsilon_2$ and $\varepsilon_3$ vary with impact parameter? What about the widths of the distributions?
* Explain your observations in terms of the geometry and the number of participants.

## <span style="color:Orange">References</span>

__[<span style="color:Red">1</span>]__ C. Loizides, J. Kamin, D. d'Enterria, "Improved Monte Carlo Glauber predictions at present and future nuclear colliders," *Phys. Rev. C* **97**, 054910 (2018); Erratum *Phys. Rev. C* **99**, 019901 (2019). [doi:10.1103/PhysRevC.97.054910](https://doi.org/10.1103/PhysRevC.97.054910), [arXiv:1710.07098](https://arxiv.org/abs/1710.07098)

__[<span style="color:Red">2</span>]__ M. L. Miller, K. Reygers, S. J. Sanders, P. Steinberg, "Glauber Modeling in High Energy Nuclear Collisions," *Ann. Rev. Nucl. Part. Sci.* **57**, 205 (2007). [doi:10.1146/annurev.nucl.57.090506.123020](https://doi.org/10.1146/annurev.nucl.57.090506.123020), [arXiv:nucl-ex/0701025](https://arxiv.org/abs/nucl-ex/0701025)

__[<span style="color:Red">3</span>]__ B. Alver, G. Roland, "Collision-geometry fluctuations and triangular flow in heavy-ion collisions," *Phys. Rev. C* **81**, 054905 (2010); Erratum *Phys. Rev. C* **82**, 039903 (2010). [doi:10.1103/PhysRevC.81.054905](https://doi.org/10.1103/PhysRevC.81.054905), [arXiv:1003.0194](https://arxiv.org/abs/1003.0194)

__[<span style="color:Red">4</span>]__ B. V. Jacak, B. Müller, "The Exploration of Hot Nuclear Matter," *Science* **337**, 310 (2012). [doi:10.1126/science.1215901](https://doi.org/10.1126/science.1215901)

__[<span style="color:Red">5</span>]__ W. Busza, K. Rajagopal, W. van der Schee, "Heavy Ion Collisions: The Big Picture, and the Big Questions," *Ann. Rev. Nucl. Part. Sci.* **68**, 339 (2018). [doi:10.1146/annurev-nucl-101917-020852](https://doi.org/10.1146/annurev-nucl-101917-020852), [arXiv:1802.04801](https://arxiv.org/abs/1802.04801)

## <span style="color:Orange">Acknowledgements</span>

* Initial version: Anne Sickles
* Revised: 2026

© Copyright 2026